# GPT-2 Inference Engine - Google Colab Setup

This notebook sets up the GPT-2 inference engine on Google Colab with T4 GPU acceleration.

**Runtime:** GPU (T4) | **Estimated time:** 15-20 minutes

## Step 1: Environment Setup

Check GPU availability and install build dependencies.

In [1]:
# Check GPU
!nvidia-smi

Fri Mar 27 16:15:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Install build dependencies
!apt-get update && apt-get install -y build-essential cmake git wget curl htop

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,473 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Pac

In [3]:
# Check CUDA version
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


## Step 2: Clone and Build GGML with CUDA Support

GGML provides the tensor computation backend with CUDA acceleration.

In [4]:
# Clone GGML (using master branch for compatibility)
%cd /content
!rm -rf ggml && git clone https://github.com/ggerganov/ggml.git --depth 1 ggml

/content
Cloning into 'ggml'...
remote: Enumerating objects: 1908, done.
remote: Counting objects: 100% (1908/1908), done.
remote: Compressing objects: 100% (1459/1459), done.
remote: Total 1908 (delta 493), reused 1478 (delta 434), pack-reused 0 (from 0)
Receiving objects: 100% (1908/1908), 2.91 MiB | 12.72 MiB/s, done.
Resolving deltas: 100% (493/493), done.


In [5]:
# Build GGML with CUDA support
%cd /content/ggml
!rm -rf build && mkdir -p build && cd build && cmake .. -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release && make -j$(nproc)

/content/ggml
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: -fopenmp (found version "4.5")
-- Found OpenMP_CXX: -fopenmp (f

In [157]:
# Verify GGML build with CUDA
!cd /content/ggml/build && ls -la src/

total 1860
drwxr-xr-x 5 root root    4096 Mar 27 16:54 .
drwxr-xr-x 7 root root    4096 Mar 27 16:16 ..
drwxr-xr-x 5 root root    4096 Mar 27 16:16 CMakeFiles
-rw-r--r-- 1 root root    3101 Mar 27 16:16 cmake_install.cmake
drwxr-xr-x 3 root root    4096 Mar 27 16:16 ggml-cpu
drwxr-xr-x 3 root root    4096 Mar 27 16:54 ggml-cuda
lrwxrwxrwx 1 root root      17 Mar 27 16:16 libggml-base.so -> libggml-base.so.0
lrwxrwxrwx 1 root root      21 Mar 27 16:16 libggml-base.so.0 -> libggml-base.so.0.9.8
-rwxr-xr-x 1 root root  747280 Mar 27 16:16 libggml-base.so.0.9.8
lrwxrwxrwx 1 root root      16 Mar 27 16:17 libggml-cpu.so -> libggml-cpu.so.0
lrwxrwxrwx 1 root root      20 Mar 27 16:17 libggml-cpu.so.0 -> libggml-cpu.so.0.9.8
-rwxr-xr-x 1 root root 1036712 Mar 27 16:17 libggml-cpu.so.0.9.8
lrwxrwxrwx 1 root root      12 Mar 27 16:54 libggml.so -> libggml.so.0
lrwxrwxrwx 1 root root      16 Mar 27 16:54 libggml.so.0 -> libggml.so.0.9.8
-rwxr-xr-x 1 root root   55184 Mar 27 16:54 libggml.so.0.9.

## Step 3: Clone Inference Engine

Clone your inference engine source files to the Colab environment.

In [158]:
# Clone the inference engine repo (includes CMakeLists.txt)
%cd /content
!rm -rf inference-engine && git clone https://github.com/nisbenz/inference-engine.git inference-engine

# Verify CMakeLists.txt exists
!ls -la /content/inference-engine/

/content
Cloning into 'inference-engine'...
remote: Enumerating objects: 360, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 360 (delta 94), reused 88 (delta 52), pack-reused 225 (from 1)
Receiving objects: 100% (360/360), 158.25 KiB | 17.58 MiB/s, done.
Resolving deltas: 100% (242/242), done.
total 120
drwxr-xr-x 4 root root  4096 Mar 27 19:43 .
drwxr-xr-x 1 root root  4096 Mar 27 19:43 ..
-rw-r--r-- 1 root root  2241 Mar 27 19:43 CMakeLists.txt
-rw-r--r-- 1 root root 90664 Mar 27 19:43 colab_gpt2_inference.ipynb
drwxr-xr-x 8 root root  4096 Mar 27 19:43 .git
-rw-r--r-- 1 root root  1064 Mar 27 19:43 LICENSE
-rw-r--r-- 1 root root    15 Mar 27 19:43 readme.md
drwxr-xr-x 2 root root  4096 Mar 27 19:43 src


In [8]:
# Upload your source files
# Use the Files panel to upload src/model.cpp, src/model.hpp, src/layers.cpp,
# src/layers.hpp, src/tokenizer.cpp, src/tokenizer.hpp, src/kv_cache.cpp,
# src/kv_cache.hpp, src/main.cpp, src/gguf_loader.cpp, src/gguf_loader.h

# Or run this cell after uploading via the Files panel
import os

src_dir = '/content/inference-engine/src'
print(f'Source files in {src_dir}:')
for f in os.listdir(src_dir):
    print(f'  {f}')

Source files in /content/inference-engine/src:
  layers.cpp
  gguf_loader.cpp
  model.cpp
  gguf_loader.h
  kv_cache.cpp
  tokenizer.cpp
  tokenizer.hpp
  layers.hpp
  main.cpp
  model.hpp
  kv_cache.hpp


## Step 4: Download GPT-2 Model (GGUF Format)

Download pre-converted GPT-2 GGUF model from HuggingFace.

**Model:** Mungert/gpt2-GGUF (F16 precision for best quality on T4 GPU)
**Config:** 12 layers, 768 hidden, 12 heads, 124M parameters

In [9]:
# Install HuggingFace Hub for downloading models
!pip install huggingface_hub -q

In [10]:
# List available GGUF files in the new repository
from huggingface_hub import hf_hub_download
from huggingface_hub import list_repo_files

print("Available GGUF files in Mungert/gpt2-GGUF:")
files = list_repo_files("Mungert/gpt2-GGUF")
for f in files:
    if f.endswith('.gguf'):
        print(f"  {f}")

Available GGUF files in Mungert/gpt2-GGUF:


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


  gpt2-bf16.gguf
  gpt2-bf16_q8_0.gguf
  gpt2-f16_q8_0.gguf
  gpt2-iq1_m.gguf
  gpt2-iq1_s.gguf
  gpt2-iq2_m.gguf
  gpt2-iq2_s.gguf
  gpt2-iq2_xs.gguf
  gpt2-iq2_xxs.gguf
  gpt2-iq3_m.gguf
  gpt2-iq3_s.gguf
  gpt2-iq3_xs.gguf
  gpt2-iq3_xxs.gguf
  gpt2-iq4_nl.gguf
  gpt2-iq4_xs.gguf
  gpt2-q2_k_m.gguf
  gpt2-q2_k_s.gguf
  gpt2-q3_k_m.gguf
  gpt2-q3_k_s.gguf
  gpt2-q4_0.gguf
  gpt2-q4_1.gguf
  gpt2-q4_k_m.gguf
  gpt2-q4_k_s.gguf
  gpt2-q5_0.gguf
  gpt2-q5_1.gguf
  gpt2-q5_k_m.gguf
  gpt2-q5_k_s.gguf
  gpt2-q6_k_m.gguf
  gpt2-q8_0.gguf
  gpt2-tq1_0.gguf
  gpt2-tq2_0.gguf


In [11]:
# Create model directory
!mkdir -p /content/gpt2-model

# Download GPT-2 F16 model (226 MB) - full precision for best quality
# For T4 GPU with 16GB VRAM, F16 fits easily and gives best results
print("Downloading GPT-2 F16 model...")
model_path = hf_hub_download(
    repo_id="Mungert/gpt2-GGUF",
    filename="gpt2-bf16.gguf",
    repo_type="model",
    local_dir="/content/gpt2-model"
)
print(f"Model downloaded to: {model_path}")

gpt2-bf16.gguf:   0%|          | 0.00/252M [00:00<?, ?B/s]

Model downloaded to: /content/gpt2-model/gpt2-bf16.gguf


In [145]:
# Download GPT-2 tokenizer files from HuggingFace
from huggingface_hub import hf_hub_download
print("Downloading tokenizer files...")
hf_hub_download(repo_id="openai-community/gpt2", filename="vocab.json", local_dir="/content/gpt2-model")
hf_hub_download(repo_id="openai-community/gpt2", filename="merges.txt", local_dir="/content/gpt2-model")

# Verify files are not empty
!ls -lh /content/gpt2-model/


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

total 243M
-rw-r--r-- 1 root root  241M Mar 27 16:56 gpt2-bf16.gguf
-rw-r--r-- 1 root root  446K Mar 27 19:27 merges.txt
-rw-r--r-- 1 root root 1018K Mar 27 19:27 vocab.json


In [ ]:
# Display all tensor names from the GGUF file for debugging
# This helps verify the tensor name mapping in model.cpp

import struct
import os

def read_gguf_tensors(gguf_path):
    """Read and display all tensor names and dimensions from a GGUF file."""
    with open(gguf_path, 'rb') as f:
        # Read header
        magic = struct.unpack('<I', f.read(4))[0]
        if magic != 0x46554747:  # 'GGUF'
            print(f"Invalid GGUF magic: {hex(magic)}")
            return []
        
        version = struct.unpack('<I', f.read(4))[0]
        tensor_count = struct.unpack('<Q', f.read(8))[0]
        metadata_count = struct.unpack('<Q', f.read(8))[0]
        
        print(f"GGUF Version: {version}")
        print(f"Tensor count: {tensor_count}")
        print(f"Metadata count: {metadata_count}")
        print("\n" + "="*80)
        print("TENSOR NAMES AND DIMENSIONS:")
        print("="*80)
        
        tensors = []
        for i in range(tensor_count):
            # Read tensor name
            name_len = struct.unpack('<Q', f.read(8))[0]
            name = f.read(name_len).decode('utf-8', errors='replace')
            
            # Read dimensions
            n_dims = struct.unpack('<I', f.read(4))[0]
            dims = []
            for _ in range(n_dims):
                dims.append(struct.unpack('<Q', f.read(8))[0])
            
            # Read type and offset
            tensor_type = struct.unpack('<I', f.read(4))[0]
            offset = struct.unpack('<Q', f.read(8))[0]
            
            tensors.append({
                'name': name,
                'dims': dims,
                'type': tensor_type
            })
            
            # Print tensor info
            type_names = {
                0: 'F32', 1: 'F16', 2: 'Q4_0', 3: 'Q4_1', 6: 'Q5_0', 7: 'Q5_1',
                8: 'Q8_0', 9: 'Q8_1', 10: 'Q2_K', 11: 'Q3_K', 12: 'Q4_K',
                13: 'Q5_K', 14: 'Q6_K', 15: 'Q8_K', 16: 'BF16', 30: 'Q8_0_ALT'
            }
            type_name = type_names.get(tensor_type, f'UNKNOWN({tensor_type})')
            print(f"{i:3d}: {name}")
            print(f"     dims={dims}, type={type_name}")
        
        return tensors

# Read and display all tensors from the model
gguf_path = '/content/gpt2-model/gpt2-bf16.gguf'
print(f"Reading GGUF file: {gguf_path}")
print(f"File size: {os.path.getsize(gguf_path) / (1024*1024):.1f} MB\n")

tensors = read_gguf_tensors(gguf_path)

# Show summary of tensor categories
print("\n" + "="*80)
print("TENSOR CATEGORY SUMMARY:")
print("="*80)

categories = {
    'Token embeddings (wte/wpe)': [],
    'Attention QKV': [],
    'Attention output proj': [],
    'Layer norms (attn)': [],
    'Layer norms (ffn)': [],
    'FFN up projection': [],
    'FFN down projection': [],
    'Final layer norm': [],
    'Other': []
}

for t in tensors:
    name = t['name']
    if 'wte' in name or 'wpe' in name or 'embed' in name:
        categories['Token embeddings (wte/wpe)'].append(name)
    elif 'attn_qkv' in name:
        categories['Attention QKV'].append(name)
    elif 'attn_output' in name:
        categories['Attention output proj'].append(name)
    elif 'attn_norm' in name:
        categories['Layer norms (attn)'].append(name)
    elif 'ffn_norm' in name:
        categories['Layer norms (ffn)'].append(name)
    elif 'ffn_up' in name:
        categories['FFN up projection'].append(name)
    elif 'ffn_down' in name:
        categories['FFN down projection'].append(name)
    elif 'ln_f' in name or 'output_norm' in name:
        categories['Final layer norm'].append(name)
    else:
        categories['Other'].append(name)

for cat, names in categories.items():
    if names:
        print(f"\n{cat}: {len(names)} tensors")
        for n in names[:3]:  # Show first 3
            print(f"  - {n}")
        if len(names) > 3:
            print(f"  ... and {len(names)-3} more")

In [159]:
# Update model path in main.cpp
# This ensures the correct model file is loaded
import re

main_cpp_path = '/content/inference-engine/src/main.cpp'
with open(main_cpp_path, 'r') as f:
    content = f.read()

# Update weights path to use F16 model
content = re.sub(
    r'std::string weights_path = "[^"]*"',
    'std::string weights_path = "/content/gpt2-model/gpt2-bf16.gguf"',
    content
)

# Update tokenizer paths
content = re.sub(
    r'std::string vocab_path = "[^"]*"',
    'std::string vocab_path = "/content/gpt2-model/vocab.json"',
    content
)
content = re.sub(
    r'std::string merges_path = "[^"]*"',
    'std::string merges_path = "/content/gpt2-model/merges.txt"',
    content
)

with open(main_cpp_path, 'w') as f:
    f.write(content)

print("Updated main.cpp with correct file paths:")
!grep -n "weights_path\|vocab_path\|merges_path" /content/inference-engine/src/main.cpp

Updated main.cpp with correct file paths:
49:    std::string weights_path = "/content/gpt2-model/gpt2-bf16.gguf";
50:    std::cout << "Loading weights from: " << weights_path << std::endl;
52:    if (!model.load_weights(weights_path)) {
57:    std::string vocab_path = "/content/gpt2-model/vocab.json";
58:    std::string merges_path = "/content/gpt2-model/merges.txt";
59:    std::cout << "Loading tokenizer from: " << vocab_path << " and " << merges_path << std::endl;
60:    if (!model.load_tokenizer(vocab_path, merges_path)) {


## Step 5: Build Custom Inference Engine

Build your own inference engine with the updated source code.

In [160]:
# Build the inference engine
# Note: GGML_DIR must point to the ggml source directory so cmake can find ggml/ggml.h
!cd /content/inference-engine && mkdir -p build && cd build && cmake .. -DCMAKE_CUDA_PATH=/usr/local/cuda -DCMAKE_CUDA_ARCHITECTURES=75 -DGGML_DIR=/content/ggml && make -j$(nproc)

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- The CUDA compiler identification is NVIDIA 12.8.93 with host compiler GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compile features - done
CMake Warning (dev) at CMakeLists.txt:9 (find_package):
  Policy CMP0146 is not set: The FindCUDA module is removed.  Run "cmake
  --help-policy CMP0146" for policy details.  Use the c

In [15]:
# Verify build output
!cd /content/inference-engine/build && ls -la bin/

ls: cannot access 'bin/': No such file or directory


## Step 6: Run Inference

Execute your inference engine with GPU acceleration.

**Usage:** `./gpt2 "prompt" [max_tokens] [temperature] [top_k]`

In [162]:
# Run with your inference engine
!cd /content/inference-engine/build && ./gpt2 "red white and" 50 0.8 40

=== GPT-2 Large Inference ===
Prompt: "red white and"
Max tokens: 50
Temperature: 0.8
Top-k: 40
GPT-2 Large model initialized
  Layers: 12
  Hidden: 768
  Heads: 12
  FFN: 3072
  Vocab: 50257
Loading weights from: /content/gpt2-model/gpt2-bf16.gguf
Loading GGUF model from: /content/gpt2-model/gpt2-bf16.gguf
Model architecture: gpt2
Config: ctx=1024 embd=768 head=12 layer=12 ffn=3072

Tensors in GGUF file:
  0: blk.0.attn_qkv.bias [2304] type=0
  1: blk.0.attn_qkv.weight [768, 2304] type=30
  2: blk.0.attn_output.bias [768] type=0
  3: blk.0.attn_output.weight [768, 768] type=30
  4: blk.0.attn_norm.bias [768] type=0
  5: blk.0.attn_norm.weight [768] type=0
  6: blk.0.ffn_norm.bias [768] type=0
  7: blk.0.ffn_norm.weight [768] type=0
  8: blk.0.ffn_up.bias [3072] type=0
  9: blk.0.ffn_up.weight [768, 3072] type=30
  10: blk.0.ffn_down.bias [768] type=0
  11: blk.0.ffn_down.weight [3072, 768] type=30
  12: blk.1.attn_qkv.bias [2304] type=0
  13: blk.1.attn_qkv.weight [768, 2304] type=30
